In [ ]:
# Copyright 2022 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

This notebook adapts Google's sample [*E2E ML on GCP: Vertex AI Training for LightGBM*](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/ml_ops/stage2/get_started_vertex_training_lightgbm.ipynb) (`get_started_vertex_training_lightgbm.ipynb` in this repo), itself a revised version of a notebook by [Rajesh Thallam](https://github.com/RajeshThallam/vertex-ai-labs/blob/main/07-vertex-train-deploy-lightgbm/vertex-train-deploy-lightgbm-model.ipynb). It replaces the Iris example with the M5 forecasting pipeline from `exercise3_m5_forecasting.ipynb`.

# M5 forecasting on Vertex AI (Gemini Enterprise Agent Platform): LightGBM custom training

## Overview

LightGBM is not one of the managed AutoML forecasting models, so it runs as a **custom training job**. The platform provisions a machine, installs our Python package into a prebuilt container, runs it and stores the outputs in Cloud Storage.

The job runs the same pipeline as section Q4 of `exercise3_m5_forecasting.ipynb`, using the same `m5_core.py` module:

1. Early stopping on fold A (d_1886–1913) picks the number of boosting rounds.
2. Every strategy is scored with WRMSSE on the untouched holdout, fold B (d_1914–1941): seasonal naive, LightGBM bottom-up, and ETS-reconciled + middle-out.
3. The model is refit on all data and forecasts d_1942–1969 with the best strategy.
4. `model.txt`, `metrics.json` and `submission.csv` are written to the job's Cloud Storage output folder.

### What changed from the Google sample

| Google sample (Iris) | This notebook (M5) |
|---|---|
| Iris loaded from scikit-learn inside the job | M5 CSVs in a Cloud Storage bucket, read through Cloud Storage FUSE (`/gcs/<bucket>/...`) |
| Small classifier, `n1-standard-4` | Feature matrix of ~20M rows: `n1-highmem-8` (52 GB RAM) |
| `scikit-learn-cpu.0-23` training image (Python 3.7) | A current Python 3.10 prebuilt CPU image; LightGBM and statsmodels installed from `setup.py` |
| FastAPI serving container + online endpoint | **Batch forecasting inside the job.** M5 forecasts are produced once per cycle for every series, so an always-on endpoint isn't needed. If on-demand forecasts are needed later, the FastAPI / custom-container section of the original sample applies unchanged with `model.txt`. |

### Costs

This tutorial uses billable components of Google Cloud: Vertex AI (Gemini Enterprise Agent Platform) training and Cloud Storage. See [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [Cloud Storage pricing](https://cloud.google.com/storage/pricing).

## Installation

In [ ]:
import os

# The Vertex AI Workbench Notebook product has specific requirements
IS_WORKBENCH_NOTEBOOK = os.getenv("DL_ANACONDA_HOME") and not os.getenv("VIRTUAL_ENV")
USER_FLAG = "--user" if IS_WORKBENCH_NOTEBOOK else ""

! pip3 install --upgrade google-cloud-aiplatform google-cloud-storage $USER_FLAG -q
! pip3 install -U lightgbm statsmodels $USER_FLAG -q

Restart the kernel after installing so the new packages are picked up.

In [ ]:
if not os.getenv("IS_TESTING"):
    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

## Before you begin

1. Select or create a Google Cloud project with billing enabled.
2. Enable the **Vertex AI (Gemini Enterprise Agent Platform) API** and the **Cloud Storage API**.
3. Make sure your account (or the notebook's service account) can run training jobs (for example *Vertex AI User*) and read/write the bucket (*Storage Object Admin*).

#### Set your project ID, region and bucket

In [ ]:
PROJECT_ID = "[your-project-id]"  # @param {type:"string"}

if PROJECT_ID == "" or PROJECT_ID is None or PROJECT_ID == "[your-project-id]":
    # Get your GCP project id from gcloud
    shell_output = ! gcloud config list --format 'value(core.project)' 2>/dev/null
    PROJECT_ID = shell_output[0]
    print("Project ID:", PROJECT_ID)

! gcloud config set project $PROJECT_ID

In [ ]:
REGION = "[your-region]"  # @param {type:"string"}

if REGION == "[your-region]":
    REGION = "us-central1"

In [ ]:
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

### Authenticate your Google Cloud account

On Vertex AI Workbench you are already authenticated. In Colab, run the next cell. Locally, set `GOOGLE_APPLICATION_CREDENTIALS` to a service-account key or use `gcloud auth application-default login`.

In [ ]:
import sys

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import auth as google_auth

    google_auth.authenticate_user()

### Create a Cloud Storage bucket

In [ ]:
BUCKET_NAME = "[your-bucket-name]"  # @param {type:"string"}

if BUCKET_NAME == "" or BUCKET_NAME is None or BUCKET_NAME == "[your-bucket-name]":
    BUCKET_NAME = PROJECT_ID + "-aip-" + TIMESTAMP
BUCKET_URI = f"gs://{BUCKET_NAME}"
DATA_URI = f"{BUCKET_URI}/m5"

**Only if your bucket doesn't already exist**: run the following cell to create it.

In [ ]:
! gcloud storage buckets create --location $REGION $BUCKET_URI

### Upload the M5 data

Download the competition files once, after accepting the competition rules on Kaggle. Use either the Kaggle CLI (needs `~/.kaggle/kaggle.json`) or a manual download. Then copy the CSVs to the bucket. The job needs `sales_train_evaluation.csv`, `calendar.csv` and `sell_prices.csv`.

In [ ]:
UPLOAD_DATA = False  # set True the first time

if UPLOAD_DATA:
    ! pip3 install -q kaggle
    ! kaggle competitions download -c m5-forecasting-accuracy -p m5_data
    ! cd m5_data && unzip -o -q m5-forecasting-accuracy.zip
    ! gcloud storage cp m5_data/sales_train_evaluation.csv m5_data/calendar.csv m5_data/sell_prices.csv $DATA_URI/

! gcloud storage ls $DATA_URI

### Initialize the Vertex AI SDK for Python

In [ ]:
import google.cloud.aiplatform as aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

#### Training container and machine type

Any prebuilt **Python ≥ 3.9 CPU** training image works: the package installs LightGBM, statsmodels and SciPy itself. Check the current list of [prebuilt training containers](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers) and change `TRAIN_IMAGE` if a newer one is available in your region.

The full M5 run (two-year training window, ~20M rows × 33 features) needs roughly 10–15 GB of RAM. `n1-highmem-8` (8 vCPUs, 52 GB) leaves comfortable headroom. More vCPUs shorten LightGBM training almost linearly.

In [ ]:
TRAIN_IMAGE = "{}-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest".format(REGION.split("-")[0])
TRAIN_COMPUTE = "n1-highmem-8"
print("Train image:", TRAIN_IMAGE, "| machine:", TRAIN_COMPUTE)

### Assemble the training package

Same layout as the Google sample:

- PKG-INFO
- README.md
- setup.cfg
- setup.py
- trainer
  - \_\_init\_\_.py
  - m5_core.py (copied from this repo; written by `exercise3_m5_forecasting.ipynb`)
  - task.py

In [ ]:
import shutil

assert os.path.exists("m5_core.py"), "m5_core.py not found: run this notebook from the repo folder"

! rm -rf custom
! mkdir -p custom/trainer
! touch custom/README.md custom/trainer/__init__.py
shutil.copy("m5_core.py", "custom/trainer/m5_core.py")

setup_cfg = "[egg_info]\n\ntag_build =\n\ntag_date = 0"
! echo "$setup_cfg" > custom/setup.cfg

setup_py = '''import setuptools

setuptools.setup(
    name="m5-lightgbm-trainer",
    version="0.1.0",
    install_requires=["lightgbm>=4.0", "statsmodels>=0.14", "scipy>=1.9", "pandas>=1.5", "numpy>=1.23"],
    packages=setuptools.find_packages(),
)
'''
with open("custom/setup.py", "w") as f:
    f.write(setup_py)

pkg_info = "Metadata-Version: 1.0\n\nName: M5 hierarchical forecasting\n\nVersion: 0.1.0\n\nSummary: LightGBM trainer for the M5 dataset\n\nPlatform: Vertex"
! echo "$pkg_info" > custom/PKG-INFO

### Create the task script

`task.py` is a thin driver around `m5_core`.

- Command-line arguments:
  - `--data-dir`: Cloud Storage folder with the CSVs, read through Cloud Storage FUSE.
  - `--model-dir`: where to write artifacts. Defaults to `AIP_MODEL_DIR`, which the service sets from `base_output_dir`.
  - `--lookback-days`, `--max-rounds`, `--learning-rate`.
- It runs the fold A / fold B / refit sequence described above and saves `model.txt`, `metrics.json` and `submission.csv`.

In [ ]:
%%writefile custom/trainer/task.py
"""Vertex AI custom training entry point for the M5 LightGBM forecaster.

Steps (same as section Q4 of exercise3_m5_forecasting.ipynb):
1. fold A: pick the number of boosting rounds by early stopping on d_1886..d_1913
2. fold B: refit and score every strategy on the untouched holdout d_1914..d_1941 (WRMSSE)
3. refit on all data, forecast d_1942..d_1969 with the best strategy
Writes model.txt, metrics.json and submission.csv to the model directory.
"""
import argparse
import json
import logging
import os
import time

import numpy as np

from trainer import m5_core as core

logging.getLogger().setLevel(logging.INFO)


def to_local(path):
    """Vertex AI mounts Cloud Storage with Cloud Storage FUSE at /gcs/<bucket>/..."""
    return "/gcs/" + path[len("gs://"):] if path.startswith("gs://") else path


parser = argparse.ArgumentParser()
parser.add_argument("--data-dir", dest="data_dir", required=True, help="folder with the Kaggle CSVs (gs:// or local)")
parser.add_argument("--model-dir", dest="model_dir", default=os.getenv("AIP_MODEL_DIR"), help="output folder")
parser.add_argument("--lookback-days", dest="lookback_days", type=int, default=728)
parser.add_argument("--max-rounds", dest="max_rounds", type=int, default=1500)
parser.add_argument("--learning-rate", dest="learning_rate", type=float, default=0.05)
args = parser.parse_args()
logging.info(args)


def main():
    t0 = time.time()
    data_dir, model_dir = to_local(args.data_dir), to_local(args.model_dir)
    os.makedirs(model_dir, exist_ok=True)

    meta, Y, calendar, prices = core.load_m5(data_dir)
    P_week, day_week = core.price_matrix(meta, calendar, prices)
    del prices
    S, info = core.build_summing_matrix(meta)
    fb = core.FeatureBuilder(meta, Y, calendar, P_week, day_week)
    closed = core.christmas_mask(calendar, Y.shape[1])
    n_days = Y.shape[1]
    holdout_end = n_days - core.HORIZON
    tune_end = holdout_end - core.HORIZON
    params = {"learning_rate": args.learning_rate}

    def window(end):
        return np.arange(end - args.lookback_days + 1, end + 1)

    logging.info("Fold A: early stopping")
    model = core.train_lgbm(fb, window(tune_end), np.arange(tune_end + 1, holdout_end + 1),
                            params=params, num_boost_round=args.max_rounds)
    rounds = model.best_iteration or args.max_rounds

    logging.info("Fold B: holdout evaluation with %d rounds", rounds)
    model = core.train_lgbm(fb, window(holdout_end), params=params, num_boost_round=rounds)
    lgb_holdout = core.predict_days(model, fb, np.arange(holdout_end + 1, n_days + 1))
    evaluator = core.WRMSSE(Y, fb.P_day, S, info, holdout_end)
    variants, _ = core.hierarchical_variants(lgb_holdout, Y, S, info, holdout_end, closed)
    scores = {name: evaluator.score(f) for name, f in variants.items()}
    scores["Seasonal naive"] = evaluator.score(core.seasonal_naive(Y, holdout_end))
    best = min(variants, key=lambda k: scores[k][0])
    for name, (total, _) in scores.items():
        logging.info("holdout WRMSSE %-40s %.4f", name, total)

    logging.info("Final refit and forecast with strategy: %s", best)
    model = core.train_lgbm(fb, window(n_days), params=params, num_boost_round=rounds)
    future = core.predict_days(model, fb, np.arange(n_days + 1, n_days + core.HORIZON + 1))
    if best != "LightGBM bottom-up":
        future = core.hierarchical_variants(future, Y, S, info, n_days, closed)[0][best]

    core.make_submission(meta, Y, future).to_csv(os.path.join(model_dir, "submission.csv"), index=False)
    model.save_model(os.path.join(model_dir, "model.txt"))
    metrics = {
        "holdout_wrmsse": {name: total for name, (total, _) in scores.items()},
        "holdout_wrmsse_by_level": {name: {int(k): float(v) for k, v in lv.items()}
                                    for name, (_, lv) in scores.items()},
        "chosen_strategy": best,
        "num_boost_round": int(rounds),
        "lookback_days": args.lookback_days,
        "learning_rate": args.learning_rate,
        "runtime_minutes": round((time.time() - t0) / 60, 1),
    }
    with open(os.path.join(model_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    logging.info("Done in %.1f min", metrics["runtime_minutes"])


if __name__ == "__main__":
    main()

#### Store the training package in your Cloud Storage bucket

In [ ]:
! rm -f custom.tar custom.tar.gz
! tar cvf custom.tar custom
! gzip custom.tar
! gcloud storage cp custom.tar.gz $BUCKET_URI/trainer_m5.tar.gz

### Create and run the custom training job

`CustomPythonPackageTrainingJob` installs the package into `container_uri` and runs `python -m trainer.task`.

**Optional local test first.** On a Workbench instance with enough RAM you can run the trainer directly: `! cd custom && python -m trainer.task --data-dir $DATA_URI --model-dir ./local_out --lookback-days 364 --max-rounds 300`.

In [ ]:
DISPLAY_NAME = "m5_lgbm_" + TIMESTAMP
MODEL_DIR = f"{BUCKET_URI}/{TIMESTAMP}"

job = aiplatform.CustomPythonPackageTrainingJob(
    display_name=DISPLAY_NAME,
    python_package_gcs_uri=f"{BUCKET_URI}/trainer_m5.tar.gz",
    python_module_name="trainer.task",
    container_uri=TRAIN_IMAGE,
    project=PROJECT_ID,
)

CMDARGS = [
    f"--data-dir={DATA_URI}",
    "--lookback-days=728",
    "--max-rounds=1500",
    "--learning-rate=0.05",
]

job.run(
    args=CMDARGS,
    replica_count=1,
    machine_type=TRAIN_COMPUTE,
    base_output_dir=MODEL_DIR,
    sync=False,
)

### Wait for the job to finish

In [ ]:
job.wait()

### Inspect the outputs

With `base_output_dir`, the service sets `AIP_MODEL_DIR` to `<base_output_dir>/model/`, so the artifacts are there.

In [ ]:
import json

import pandas as pd

ARTIFACTS = f"{MODEL_DIR}/model"
! gcloud storage ls $ARTIFACTS
! gcloud storage cp $ARTIFACTS/metrics.json $ARTIFACTS/submission.csv .

metrics = json.load(open("metrics.json"))
print("Chosen strategy:", metrics["chosen_strategy"], "| rounds:", metrics["num_boost_round"],
      "| runtime:", metrics["runtime_minutes"], "min")
display(pd.Series(metrics["holdout_wrmsse"], name="holdout WRMSSE").sort_values().to_frame())
display(pd.DataFrame(metrics["holdout_wrmsse_by_level"]).round(3))
display(pd.read_csv("submission.csv").head())

### Optional: register the model

Registering `model.txt` in the Model Registry gives versioning and lineage. Serving it needs a custom container (see the FastAPI section of the original Google sample), because there is no prebuilt LightGBM prediction image.

In [ ]:
REGISTER_MODEL = False

if REGISTER_MODEL:
    model = aiplatform.Model.upload(
        display_name=f"m5-lgbm-{TIMESTAMP}",
        artifact_uri=ARTIFACTS,
        serving_container_image_uri="[your-region]-docker.pkg.dev/[your-project]/prediction/lightgbm-cpu:latest",
        description="M5 global LightGBM (Tweedie) forecaster",
    )
    print(model.resource_name)

### Productionising

- **Schedule.** Run this job once per forecast cycle, for example with Vertex AI Pipelines or a Cloud Scheduler trigger. Point `--data-dir` at the latest data snapshot.
- **Monitor.** Track `metrics.json` (holdout WRMSSE overall and by level) across runs. Alert when a level degrades or the chosen strategy changes.
- **Tune.** Hyperparameter tuning jobs can sweep `--learning-rate` and other arguments, with the holdout WRMSSE reported as the metric.

# Cleaning up

In [ ]:
delete_bucket = False

try:
    job.delete()
except Exception as e:
    print(e)

if delete_bucket or os.getenv("IS_TESTING"):
    ! gcloud storage rm --recursive $BUCKET_URI